In [5]:
import os
from dotenv import load_dotenv
from nltk.data import retrieve

# from data_ingest_parsing.data_ingestion import documents

load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv('OPENAI_API_KEY')

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain_core.documents import Document

## vector store
from langchain_community.vectorstores import Chroma

import numpy as np
from typing import List

In [15]:
# RAG Architecture Overview
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



In [16]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are three main
    types of machine learning: supervised learning, unsupervised learning, and reinforcement
    learning. Supervised learning uses labeled data to train models, while unsupervised
    learning finds patterns in unlabeled data. Reinforcement learning learns through
    interaction with an environment using rewards and penalties.
    """,

    """
    Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks.
    These networks are inspired by the human brain and consist of layers of interconnected
    nodes. Deep learning has revolutionized fields like computer vision, natural language
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers
    excel at sequential data processing.
    """,

    """
    Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language.
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis,
    machine translation, and question answering. Modern NLP heavily relies on transformer
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand
    context and relationships between words in text.
    """
]

sample_docs

['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn\n    and improve from experience without being explicitly programmed. There are three main\n    types of machine learning: supervised learning, unsupervised learning, and reinforcement\n    learning. Supervised learning uses labeled data to train models, while unsupervised\n    learning finds patterns in unlabeled data. Reinforcement learning learns through\n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionized fields like computer vision, natural language\n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly\n    effective for image 

In [6]:
import tempfile
temp_dir=tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt",'w') as f:
        f.write(doc)

print(f"sample doc created in : {temp_dir}")

sample doc created in : C:\Users\asus\AppData\Local\Temp\tmp6xnms6wu


In [17]:
# doc loading
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "data",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'}
)
documents=loader.load()

print(f"Loaded {len(documents)} documents")
print(f"\n First doc preview")
print(documents)

Loaded 3 documents

 First doc preview
[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='\n    Machine Learning Fundamentals\n    \n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    '), Document(metadata={'source': 'data\\doc_1.txt'}, page_content='\n    Deep Learning and Neural Networks\n    \n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has rev

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Document Splitting
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=[" "]
)

In [19]:
chunks = text_splitter.split_documents(documents)

In [20]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n    \n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n    \n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of laye

In [21]:
embeddings=OpenAIEmbeddings()
sample_text="Machine learning is a subset of artificial intelligence that enables system"

In [22]:
vector=embeddings.embed_query(sample_text)
vector

[-0.025024907663464546,
 -0.005532559007406235,
 -0.013833064585924149,
 -0.0028296420350670815,
 -0.014726811088621616,
 0.03404241055250168,
 -0.006889853160828352,
 -0.005018988158553839,
 -0.020676229149103165,
 -0.021676691249012947,
 0.020035933703184128,
 0.016007402911782265,
 -0.0031381179578602314,
 -0.010978410951793194,
 -0.0008358032209798694,
 0.004772207234054804,
 0.009851222857832909,
 0.025225000455975533,
 0.014926903881132603,
 -0.011058447882533073,
 -0.018208421766757965,
 0.018715322017669678,
 0.007670213934034109,
 -0.023090679198503494,
 -0.0014389988500624895,
 -0.02498488873243332,
 0.008917457424104214,
 -0.041779324412345886,
 -0.006763128098100424,
 0.010044645518064499,
 0.01329281460493803,
 -0.014219909906387329,
 -0.020276043564081192,
 -0.026652326807379723,
 -0.02499822899699211,
 -0.014593415893614292,
 -0.0011538669932633638,
 0.0024811476469039917,
 0.014046496711671352,
 -0.0018225095700472593,
 0.021329864859580994,
 0.017981650307774544,
 -0.0

### CHROMA DB

In [23]:
persist_dir="./chroma_db"
# initialize chromadb with openai embeddings
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=OpenAIEmbeddings(),
    persist_directory=persist_dir,
    collection_name="rag_collection",
)
print(vectorstore._collection.count())
print(persist_dir)

25
./chroma_db


### Test similarity Search

In [24]:
query="what is nlp"
similar_docs=vectorstore.similarity_search(query,k=1)
similar_docs

[Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n    \n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.')]

In [25]:
### Advanced similarity Search with score
vectorstore.similarity_search_with_score(query,k=3)

[(Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n    \n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
  0.30609583854675293),
 (Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n    \n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and 

In [26]:
from langchain_openai import ChatOpenAI

llm=ChatOpenAI(
    model_name="gpt-3.5-turbo"
)


In [27]:
llm

ChatOpenAI(profile={'max_input_tokens': 16385, 'max_output_tokens': 4096, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000017CA3A23F40>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000017CA3A646D0>, root_client=<openai.OpenAI object at 0x0000017CA3A23EE0>, root_async_client=<openai.AsyncOpenAI object at 0x0000017CA3A64640>, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [28]:
text_response=llm.invoke("what is llm?")

In [29]:
text_response

AIMessage(content='LLM stands for Master of Laws, which is a postgraduate degree in law that allows individuals to specialize in a specific area of law. It is typically one year in duration and is designed for those who have already completed a law degree and wish to further their legal education and career opportunities.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 12, 'total_tokens': 70, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DiO9vCAsslZxG2GJNReGC4wwXrRdX', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e50c6-db00-71c0-a4a2-b87771f492c1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_token

In [30]:
from langchain.chat_models.base import init_chat_model
llm=init_chat_model("openai:gpt-3.5-turbo")
llm.invoke("what is ml?")

AIMessage(content='ML, or machine learning, is a type of artificial intelligence that enables computers to learn and improve from experience without being explicitly programmed. In ML, algorithms are used to analyze patterns in data, make decisions and predictions, and improve over time. This technology is used in various applications, such as image and speech recognition, recommendation systems, and autonomous vehicles.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 11, 'total_tokens': 81, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DiOA4lrneaLJGPVzXSovosSgQWSVH', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e50

# Modern Rag Chain

In [31]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableMap, RunnableLambda
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain.agents.middleware import dynamic_prompt, ModelRequest

In [32]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [33]:
retriever=vectorstore.as_retriever(
    search_kwargs={"k":3}
)
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000017CA1683E80>, search_kwargs={'k': 3})

In [34]:
system_prompt = """
    YOu are an assistant for question-answer tasks.
    Use the following pieces of retrieved context to answer the question.
    If you don't know the answer, just say that you don't know.
    Use three sentences max and keep the answer concise.

    Context: {context}
"""

prompt=ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [35]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\n    YOu are an assistant for question-answer tasks.\n    Use the following pieces of retrieved context to answer the question.\n    If you don't know the answer, just say that you don't know.\n    Use three sentences max and keep the answer concise.\n\n    Context: {context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [36]:
# create a doc chain
chain = prompt | llm


In [37]:
chain

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\n    YOu are an assistant for question-answer tasks.\n    Use the following pieces of retrieved context to answer the question.\n    If you don't know the answer, just say that you don't know.\n    Use three sentences max and keep the answer concise.\n\n    Context: {context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOpenAI(profile={'max_input_tokens': 16385, 'max_output_tokens': 4096, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'image_u

In [38]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs) if docs else "No relevant context found."

SYSTEM = """
You are an assistant for question-answer tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences max and keep the answer concise.

Context:
{context}
"""
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("human", "{input}")
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

chain = (
    {
        # feed the user question to the retriever
        "context": itemgetter("input") | retriever | RunnableLambda(format_docs),
        # also pass the original input into the prompt
        "input": itemgetter("input"),
    }
    | prompt
    | llm
    | StrOutputParser()
)

answer = chain.invoke({"input": "Summarize the policy on reimbursements."})
print(answer)


I don't know.


In [39]:
response = chain.invoke({
    "input": "What is deep learning"
})

print(response)


Deep learning is a subset of machine learning that utilizes artificial neural networks inspired by the human brain. It involves layers of interconnected nodes and has significantly impacted fields such as computer vision, natural language processing, and speech recognition.


In [40]:
# pip install langchain-core langchain-openai

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# === you already have this ===
# documents: list[Document] from DirectoryLoader

def format_docs(docs):
    return "\n\n".join(getattr(d, "page_content", str(d)) for d in docs) if docs else "No relevant context found."

SYSTEM = """You are an assistant for question-answer tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences max and keep the answer concise.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("human", "{input}")
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ✅ Pass the docs at invoke-time
chain = (
    RunnablePassthrough.assign(context=lambda x: format_docs(x["context"]))
    | prompt
    | llm
    | StrOutputParser()
)

answer = chain.invoke({
    "input": "what is deep learning?",
    "context": documents   # <-- your loaded docs go here
})

print(answer)


Deep learning is a subset of machine learning that is based on artificial neural networks, which are inspired by the human brain. It involves layers of interconnected nodes and has significantly advanced fields such as computer vision, natural language processing, and speech recognition. Deep learning techniques, like Convolutional Neural Networks (CNNs) and Recurrent Neural Networks (RNNs), are particularly effective for processing images and sequential data, respectively.


In [41]:
for d in documents:
    print(getattr(d,"page_content",str(d)))


    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    

    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image proce

Create RAG chain alternative - Using LCEL(Langchain Expression Language)

In [42]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [43]:
custom_prompt = ChatPromptTemplate.from_template("""
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Provide specific details from the context to support your answer.

Context:
{context}

Question: {question}

Answer:
""")
custom_prompt


ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:\n"), additional_kwargs={})])

In [44]:
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000017CA1683E80>, search_kwargs={'k': 3})

In [45]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs) if docs else "No relevant context found."

In [48]:
rag_chain_lcel = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | custom_prompt
    | llm
    | StrOutputParser()
)

In [57]:
res = rag_chain_lcel.invoke("What is DL")

In [58]:
print(res)

DL stands for Deep Learning, which is a subset of machine learning based on artificial neural networks. These networks are inspired by the human brain and consist of layers of interconnected nodes. Deep learning has significantly impacted various fields, including computer vision, natural language processing, and speech recognition.


In [68]:
def query_rag_lcel(question):
    print(f"question: {question}")
    print("-"*50)

    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")

    source = retriever.invoke(answer)
    for s in source:
        print(s.page_content)


In [69]:
query_rag_lcel(input())

question: what is ml?
--------------------------------------------------
Answer: Machine learning (ML) is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It involves three main types: supervised learning, which uses labeled data to train models; unsupervised learning, which finds patterns in unlabeled data; and reinforcement learning, which learns through interactions with an environment.
Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through
Machine Learning Fundamentals
    
    M